# Run the IDS-XAI project on Google Colab

This notebook reruns the whole project on a Colab runtime: environment setup, data checks, unit tests, the benchmark, the robustness audit, the paper build, and the integrity verifier.

**Before you start, put two files in Google Drive** (default folder `MyDrive/ids-xai/`):

| File | How to make it (on the computer that has the project) |
|---|---|
| `ids-xai.zip` | In the project folder: `git archive -o ids-xai.zip HEAD` |
| `ids-xai-data.zip` | In the folder that contains `data/raw/`: `zip -r ids-xai-data.zip data/raw` |

`ids-xai-data.zip` holds the four parquet files (about 220 MB) declared in `config.yaml`. The raw datasets are not redistributed with the code, so obtain them under the CIC and UNSW licence terms.

**Run time.** A full run took 28 min (benchmark) + 16 min (audit) on a 16-core machine. A free Colab runtime has about 2 vCPUs, so expect roughly 2-4 hours. Set `QUICK = True` below for a 15-30 minute smoke run (one model seed, 40 flows) that exercises every step but does not reproduce the paper's numbers.

In [ ]:
# Settings - edit these if your Drive layout differs.
DRIVE_FOLDER = '/content/drive/MyDrive/ids-xai'
PROJECT_ZIP = f'{DRIVE_FOLDER}/ids-xai.zip'
DATA_ZIP = f'{DRIVE_FOLDER}/ids-xai-data.zip'
PROJECT_DIR = '/content/ids-xai'
QUICK = False  # True: fast smoke run with a reduced configuration

## 1. Mount Drive and unpack the project and data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, pathlib
pathlib.Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PROJECT_ZIP) as archive:
    archive.extractall(PROJECT_DIR)
with zipfile.ZipFile(DATA_ZIP) as archive:
    archive.extractall(PROJECT_DIR)  # creates data/raw/*.parquet
os.chdir(PROJECT_DIR)
print(sorted(os.listdir('data/raw')))

## 2. Install the pinned packages

The pins in `requirements.txt` target Python 3.12. If Colab reports that numpy or pandas were replaced, use **Runtime > Restart session** once, then continue from the next section (re-run the settings cell first).

In [ ]:
import sys
print('Python', sys.version)
%pip install -q -r requirements.txt

## 3. Check the data matches the recorded run

In [ ]:
import os, json, hashlib
os.chdir(PROJECT_DIR)
manifest = json.load(open('outputs/run_manifest.json'))
for recorded_path, meta in manifest['files'].items():
    path = recorded_path.replace('\\', '/')
    digest = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    status = 'OK' if digest == meta['sha256'] else 'MISMATCH - results will differ'
    print(f'{path}: {status}')

## 4. Unit tests

In [ ]:
!cd {PROJECT_DIR} && python -B -m unittest discover -s tests

## 5. Run the experiments

The full run writes to `outputs/` and is what the paper and verifier read. Quick mode writes to `outputs_quick/` with a reduced configuration and skips the paper and verifier, whose checks assume the full protocol.

In [ ]:
import yaml
os.chdir(PROJECT_DIR)
config_path = 'config.yaml'
if QUICK:
    config = yaml.safe_load(open('config.yaml'))
    config['output_dir'] = 'outputs_quick'
    config['predictive_seeds'] = [42]
    config['evaluation'].update(
        explanation_samples=40, explanation_seeds=[42], lime_samples=150,
        audit_samples=20, lime_budget_samples=6,
    )
    config['tuning'] = {'validation_size': config['tuning']['validation_size']}  # skip the grid search
    yaml.safe_dump(config, open('config_quick.yaml', 'w'), sort_keys=False)
    config_path = 'config_quick.yaml'
print('Using', config_path)

In [ ]:
!cd {PROJECT_DIR} && python -B run_benchmark.py --config {config_path}

In [ ]:
!cd {PROJECT_DIR} && python -B run_extended_audit.py --config {config_path}

## 6. Build the paper and verify every artifact (full run only)

In [ ]:
if not QUICK:
    !cd {PROJECT_DIR} && python -B build_paper.py && python -B verify_outputs.py
else:
    print('Quick mode: inspect outputs_quick/ instead; the paper needs the full protocol.')

## 7. Save results to Drive

In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d-%H%M')
results = 'outputs_quick' if QUICK else 'outputs'
target = f'{DRIVE_FOLDER}/results-{stamp}'
shutil.copytree(f'{PROJECT_DIR}/{results}', f'{target}/{results}',
                ignore=shutil.ignore_patterns('models'))
if not QUICK:
    shutil.copytree(f'{PROJECT_DIR}/output', f'{target}/output')
print('Saved to', target)

## 8. Explore the results

Open `PROJECT_WALKTHROUGH.ipynb` from `/content/ids-xai` (File > Open notebook > Upload, or `from google.colab import files`) to read the measured results step by step. It finds the project in `/content/ids-xai` automatically.

To explain one alert with a trained model (models are kept only in the runtime, under `outputs/models/`):

```bash
python explain_alert.py --model outputs/models/UNSW_NB15_binary_xgboost.joblib \
    --input path/to/flows.csv --row 0 --output outputs/alert_reports/alert_0
```